# Create CONAHCYT SNII Awards (GRANT/FELLOWSHIP PATTERN, CKAN-discovery + bulk CSV)

Ingest of Mexico's **Sistema Nacional de Investigadores (SNII)** — the national researcher recognition + stipend program administered by CONAHCYT/SECIHTI (program budget code S191). 13 years of annual researcher rolls (2012-2024), ~30-40K researchers per year, dedup'd to one row per appointment cycle. Fills the Mexico Latam gap left by the existing Argentina (#43), Brazil FAPESP (#19), Chile ANID (#35), Colombia MinCiencias (#52) entries.

**Funder name history** — CONACYT (since 1970) → **CONAHCYT** (renamed 2023 under President López Obrador) → **SECIHTI** (renamed 2024 under President Sheinbaum). All three names refer to the same institutional entity. OpenAlex has only the original CONACYT-era display name on file (`F4320321739`, *"Consejo Nacional de Ciencia y Tecnología"*). One funder_id covers all eras — unlike MinCiencias #52 (dual id year-bounded with COLCIENCIAS), CONAHCYT/SECIHTI doesn't have a separate OpenAlex funder row to bridge. The current publishing host `repodatos.atdt.gob.mx/api_update/secihti/...` reflects the SECIHTI rebrand.

**Awarding body:** Consejo Nacional de Ciencia y Tecnología — `F4320321739` (MX)

**Source authority** — Mexico's national open-data portal `www.datos.gob.mx` (CKAN), method #1 on the runbook ladder. NOT an aggregator: CONAHCYT publishes here directly via Mexico's Agencia de Transformación Digital y Telecomunicaciones (ATDT).

**Schema choices:**
- **One row per appointment cycle**, not per disbursement year. The raw CSVs publish annual snapshots where the same researcher appears in 3 consecutive years for one 3-year SNII appointment. Dedup key = `(cvu, nivel_distincion, fecha_inicio_vig)`, keeping the most recent annual record per key. This follows the Kyle b121826 MinCiencias precedent: *"dedup by the column we ship as amount"*.
- **`amount` = monto_anual + monto_anual_adicional**, currency MXN — the annual stipend at the most-recently-reported year for that appointment. Published on every row; full §6.7 check applies (NOT waived).
- **`funder_scheme` = SNII level** — `'SNII Investigador Nivel 1'` / `'Nivel 2'` / `'Nivel 3'` / `'Candidato a Investigador'` / `'Investigador Emérito'`. Five distinct schemes.
- **`funding_type`** — SQL CASE on snii_level: `'fellowship'` for Candidato (early-career), `'research'` for Niveles 1-3 + Emérito (active researchers).
- **`lead_investigator`** — Spanish double-surname convention: `family_name = primer_apellido` (paternal surname is the family name in Spanish naming); `given_name = nombre`. The maternal surname (`segundo_apellido`) is preserved as `maternal_surname` metadata but not used in the family_name field. Affiliation = `inst_adscrip` with `country='MX'`.
- **Dates**: `start_date` / `end_date` parsed via TRY_TO_DATE on the script-normalized ISO strings (the source CSV mixes `YYYY-MM-DD` and `D/M/YYYY` Spanish formats — handled in the upload script).
- **`declined`** always False — SNII publishes only awarded researchers, no declined records.

**Scope decision**: this PR is **S191 SNII researchers only**. Two other CONAHCYT programs published on the same CKAN portal are deferred to follow-up:
- **S190 Becas** (postgraduate scholarships, ~6 yearly CSVs) — recipient name is NOT exposed in the CSVs (anonymous beneficiary records, only institution + amount). Would land as institution-level award rows; tracker entry documents the limitation.
- **P001 Cátedras CONAHCYT** (research-chair positions, 10 yearly CSVs) — 86-column payroll/compensation data for CONAHCYT-employed researchers posted to host universities, not grant data. HR transparency disclosure, not awards.

**Prerequisites**: Run `scripts/local/conahcyt_to_s3.py` first to download all 13 SNII CSVs (~10MB each) via CKAN-discovered URLs, dedup, write parquet, upload to S3.

**Data source**: https://www.datos.gob.mx/api/3/action/package_show?id=programas_presupuestarios_conahcyt
**S3 location**: `s3a://openalex-ingest/awards/conahcyt/conahcyt_snii.parquet`

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.conahcyt_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/conahcyt/conahcyt_snii.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.conahcyt_raw;

## Step 1.5: Inspect raw + money/currency scan + dedup verification

The upload script applies the dedup rule (`cvu, nivel_distincion, fecha_inicio_vig`). This step sanity-checks that the source CSVs really are tranche-shape (multiple annual rows per appointment) and that the dedup collapsed them correctly.

Expected post-dedup row count: ~150K-200K unique appointment cycles (from ~470K raw rows summed across 13 yearly CSVs). amount_mxn should be ~95-100% populated (SNII publishes amounts on every row). Currency = `'MXN'` only.

Key source columns: `funder_award_id`, `cvu`, `last_reported_year`, `researcher_full_name`, `given_name`, `family_name`, `maternal_surname`, `snii_level`, `snii_level_label`, `grado_estudios`, `institution`, `entidad_federativa`, `area_conocimiento`, `start_date`, `end_date`, `amount_mxn`, `currency`, `tipo_apoyo`, `otros_apoyos`, `declined`.

In [ ]:
%sql
DESCRIBE openalex.awards.conahcyt_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.conahcyt_raw LIMIT 5;

In [ ]:
%sql
-- §1.5 money-shape scan — confirm amount_mxn is in the expected MXN-stipend range (~50K-500K MXN/yr).
SELECT
    MIN(TRY_CAST(amount_mxn AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount_mxn AS DOUBLE)) AS max_amount,
    AVG(TRY_CAST(amount_mxn AS DOUBLE)) AS avg_amount,
    COUNT(amount_mxn) AS non_null,
    COUNT(*) AS total_rows
FROM openalex.awards.conahcyt_raw;

In [ ]:
%sql
-- Currency check — should be only 'MXN' or NULL.
SELECT currency, COUNT(*) AS rows FROM openalex.awards.conahcyt_raw GROUP BY currency;

In [ ]:
%sql
-- Dedup sanity: distinct (cvu, snii_level, start_date) keys should equal row count.
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT CONCAT(cvu, '|', snii_level, '|', COALESCE(start_date,''))) AS distinct_appointment_keys,
    COUNT(*) - COUNT(DISTINCT CONCAT(cvu, '|', snii_level, '|', COALESCE(start_date,''))) AS dedup_residuals_should_be_zero
FROM openalex.awards.conahcyt_raw;

## Step 1.6: Fail-fast — verify CONAHCYT funder row exists

In [ ]:
%sql
-- Must return exactly 1 row. OpenAlex has only the original CONACYT-era name on file.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320321739;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.conahcyt_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320321739  -- CONAHCYT / SECIHTI (CONACYT-era display name in OpenAlex)
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    CONCAT(n.snii_level_label, ' \u2014 ', n.researcher_full_name) AS display_name,
    CASE
        WHEN n.area_conocimiento IS NOT NULL AND n.institution IS NOT NULL
            THEN CONCAT('SNII appointment in ', n.area_conocimiento, ' at ', n.institution, '.')
        WHEN n.area_conocimiento IS NOT NULL
            THEN CONCAT('SNII appointment in ', n.area_conocimiento, '.')
        WHEN n.institution IS NOT NULL
            THEN CONCAT('SNII appointment at ', n.institution, '.')
        ELSE 'SNII appointment.'
    END AS description,
    f.funder_id,
    n.funder_award_id,
    TRY_CAST(n.amount_mxn AS DOUBLE) AS amount,
    n.currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    -- 'Candidato a Investigador' = early-career fellowship; all other SNII levels = research.
    CASE
        WHEN UPPER(n.snii_level) = 'C' THEN 'fellowship'
        ELSE 'research'
    END AS funding_type,
    n.snii_level_label AS funder_scheme,
    'conahcyt_snii_ckan' AS provenance,
    TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(n.end_date, 'yyyy-MM-dd') AS end_date,
    TRY_CAST(SUBSTRING(n.start_date, 1, 4) AS INT) AS start_year,
    TRY_CAST(SUBSTRING(n.end_date, 1, 4) AS INT) AS end_year,
    struct(
        n.given_name AS given_name,
        n.family_name AS family_name,
        CAST(NULL AS STRING) AS orcid,
        TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS role_start,
        struct(
            n.institution AS name,
            CAST('MX' AS STRING) AS country,  -- SNII members are by definition Mexico-affiliated
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url,  -- NULL: CONAHCYT does not expose per-SNII-researcher public landing pages
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.conahcyt_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.cvu IS NOT NULL
  AND n.snii_level IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 83

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'conahcyt_snii_ckan' AND priority = 83;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    83 AS priority  -- CONAHCYT SNII priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.conahcyt_awards;

## Step 6: Verification

Full §6.1-6.7 — CONAHCYT amount-coverage is NOT waived. Expect:
- Row count ~150K-200K unique appointment cycles (dedup'd from ~470K raw researcher-years)
- ~95-99% `pct_amount` (SNII publishes amounts on every row)
- `currencies = [MXN]`
- Year range 2012-2024 covered, with the bulk in 2019-2024
- 5 distinct funder_schemes (SNII Nivel 1/2/3/Candidato/Emérito)
- `funding_type` ~95% research, ~5% fellowship (Candidatos)

In [ ]:
%sql
SELECT COUNT(*) AS total_conahcyt_award_rows FROM openalex.awards.conahcyt_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.conahcyt_awards;

In [ ]:
%sql
-- §6.3 Data completeness
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) AS pct_title,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) AS pct_dates,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) AS pct_description
FROM openalex.awards.conahcyt_awards;

In [ ]:
%sql
-- §6.7 amount + currency fail-fast (NOT waived).
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount_mxn,
    ROUND(MAX(amount), 0) AS max_amount_mxn,
    ROUND(AVG(amount), 0) AS avg_amount_mxn
FROM openalex.awards.conahcyt_awards;

In [ ]:
%sql
-- SNII level distribution — expect Nivel 1 largest, then 2, 3, Candidato, Emérito smallest.
SELECT funder_scheme AS snii_level, funding_type, COUNT(*) AS rows,
       ROUND(SUM(amount) / 1e6, 1) AS total_mxn_millions,
       ROUND(AVG(amount), 0) AS avg_amount_mxn
FROM openalex.awards.conahcyt_awards
GROUP BY funder_scheme, funding_type
ORDER BY rows DESC;

In [ ]:
%sql
-- Sample
SELECT id, SUBSTRING(display_name, 1, 80) AS title,
       funder_scheme AS level, funding_type, amount, start_year, end_year,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       SUBSTRING(lead_investigator.affiliation.name, 1, 50) AS institution
FROM openalex.awards.conahcyt_awards
WHERE amount IS NOT NULL
ORDER BY amount DESC
LIMIT 12;

In [ ]:
%sql
-- Year distribution
SELECT start_year, COUNT(*) AS rows, ROUND(SUM(amount)/1e6, 1) AS total_mxn_millions
FROM openalex.awards.conahcyt_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year DESC
LIMIT 25;

In [ ]:
%sql
-- Funder struct populated correctly?
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.conahcyt_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;